## Notebook to learn to play with tif images

In [1]:
import sys
import random
import importlib as imp
import numpy as np
import matplotlib.pyplot as plt

import experiment_settings
import build_model
import train_model
import build_data

import tensorflow as tf

# things to do next
# 1. make a master list from which to train / predict including a buffer zone around coastal regions
# 2. predict all tiles that have at least ONE value within them (this likely will not be a waste of space / time)

In [2]:
print(f"python version = {sys.version}")
print(f"numpy version = {np.__version__}")
print(f"tensorflow version = {tf.__version__}")  

# tf.config.set_visible_devices([], "GPU")  # turn-off tensorflow-metal if it is on
# print(tf.config.list_physical_devices('GPU'))

python version = 3.10.10 | packaged by conda-forge | (main, Mar 24 2023, 20:12:31) [Clang 14.0.6 ]
numpy version = 1.23.2
tensorflow version = 2.10.0


In [3]:
# GET SETTINGS
EXP_NAME = "exp0"
settings = experiment_settings.get_settings(EXP_NAME)

# SET RANDOM SEEDS
np.random.seed(settings["rng_seed"])
random.seed(settings["rng_seed"])
tf.random.set_seed(settings["rng_seed"])

In [4]:
# LOAD THE DATA
imp.reload(build_data)

(tagyear_train, 
 taglat_train, 
 taglon_train,
 tagyear_val, 
 taglat_val, 
 taglon_val, 
 ) = build_data.make_sample_list(settings)

tfds_train = build_data.build_tf_dataset(settings, tagyear_train, taglat_train, taglon_train, settings["batch_size"])
tfds_val = build_data.build_tf_dataset(settings, tagyear_val, taglat_val, taglon_val, settings["batch_size"])

batch_shape = np.shape(next(tfds_val.as_numpy_iterator())[0])
print(f"{batch_shape = }")

output region shape = (327, 327)
ntrain = 6400, nval = 800
Metal device set to: Apple M1 Max

systemMemory: 64.00 GB
maxCacheSize: 24.00 GB

batch_shape = (32, 114, 114, 6)


In [5]:
# TRAIN THE MODEL
imp.reload(build_model)
imp.reload(train_model)

tf.keras.backend.clear_session()
model = build_model.build_model(settings, input_shape=batch_shape[1:])

model, fit_summary, history, settings = train_model.train_model(settings, model, tfds_train, tfds_val)

fit_summary

Epoch 1/1000
200/200 [==============================] - 43s 210ms/step - loss: 0.0298 - mae: 0.1240 - val_loss: 0.0217 - val_mae: 0.1096 - lr: 0.0010
Epoch 2/1000
200/200 [==============================] - 43s 213ms/step - loss: 0.0178 - mae: 0.0973 - val_loss: 0.0199 - val_mae: 0.1013 - lr: 9.0484e-04
Epoch 3/1000
200/200 [==============================] - 43s 215ms/step - loss: 0.0177 - mae: 0.0961 - val_loss: 0.0198 - val_mae: 0.0991 - lr: 8.1873e-04
Epoch 4/1000
200/200 [==============================] - 43s 215ms/step - loss: 0.0160 - mae: 0.0932 - val_loss: 0.0201 - val_mae: 0.1047 - lr: 7.4082e-04
Epoch 5/1000
200/200 [==============================] - 43s 212ms/step - loss: 0.0145 - mae: 0.0897 - val_loss: 0.0210 - val_mae: 0.1080 - lr: 6.7032e-04
Epoch 6/1000
200/200 [==============================] - 43s 215ms/step - loss: 0.0140 - mae: 0.0880 - val_loss: 0.0197 - val_mae: 0.1004 - lr: 6.0653e-04
Epoch 7/1000
200/200 [==============================] - 43s 215ms/step - loss: 0

{'elapsed_time': 518.5569248199463,
 'best_epoch': 8,
 'loss_train': 0.010707476176321507,
 'loss_valid': 0.01901078037917614,
 'mae_train': 0.07807397842407227,
 'mae_valid': 0.09683798253536224}